# Support/ops agent: one store, every retrieval tool

A useful agent reaches for the *right* kind of search per question — semantic for
vague needs, exact keyword for a named product, SQL for counts and filters. A
LangGraph agent gets all of them from **one** Infino store: vector, BM25, and SQL
search as tools, plus a semantic LLM cache and multi-turn memory backed by the
same engine.

**This example needs an LLM key** (the agent is tool-calling) — see the README.
Building and indexing the store is key-free; the agent and cache need the key.

In [1]:
import sys
import shutil
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # examples/ root, where _shared/ lives

import infino
import pyarrow as pa
from langchain.agents import create_agent
from langchain_core.globals import set_llm_cache
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver

from langchain_infino import InfinoSemanticCache, InfinoVectorStore

from _shared.lc import MiniLMEmbeddings, chat_model
from _shared.embedding import DIM, METRIC
from _shared.loaders import load_amazon
from _shared.sql import query

DB_DIR = "./agent_data"
TABLE = "catalog"
shutil.rmtree(DB_DIR, ignore_errors=True)  # start fresh each run

embeddings = MiniLMEmbeddings()
llm = chat_model()
db = infino.connect(DB_DIR)
if llm is None:
    print("This example needs an LLM key (tool-calling agent). See the README.")

## Build the catalog store

One `InfinoVectorStore` over a product catalog, with `price` and `rating`
promoted to scalar columns so SQL and filters can use them. We append a few
known featured items so the demo questions have clear targets.

In [2]:
products = load_amazon(n=300)
store = InfinoVectorStore.from_texts(
    [p["text"] for p in products], embeddings,
    metadatas=[{"title": p["title"], "category": p["category"],
                "price": p["price"], "rating": p["rating"]} for p in products],
    connection=db, table_name=TABLE, dim=DIM, metric=METRIC,
    metadata_columns=[pa.field("price", pa.float64(), nullable=False),
                      pa.field("rating", pa.float64(), nullable=False)],
)

FEATURED = [
    ("NovaSound Z9 wireless running earbuds, sweatproof, 30h battery",
     "NovaSound Z9", 49.0, 4.7),
    ("HydroPeak insulated steel water bottle, 32oz, keeps drinks cold 24h",
     "HydroPeak 32oz Bottle", 24.0, 4.6),
    ("ZenFlow non-slip yoga mat, 6mm cushioned eco TPE",
     "ZenFlow Yoga Mat", 33.0, 4.8),
    ("LumaDesk LED desk lamp, adjustable warmth, USB-C charging",
     "LumaDesk LED Lamp", 39.0, 4.4),
]
store.add_texts(
    [f[0] for f in FEATURED],
    metadatas=[{"title": f[1], "category": "Featured", "price": f[2],
                "rating": f[3]} for f in FEATURED],
    ids=[f"sku-{i}" for i in range(len(FEATURED))],
)
print("indexed", query(db, f"SELECT COUNT(*) AS n FROM {TABLE}")["n"][0], "products")

indexed 304 products


## Tools — three retrieval modes over the one store

Each tool is a thin wrapper over a different Infino retrieval mode. The agent
reads the docstrings and picks: `semantic_search` for intent, `keyword_search`
for a named product, `query_catalog` for counts and filters — all three over the
same table.

In [3]:
@tool
def semantic_search(need: str) -> str:
    """Find products by meaning or intent (e.g. 'something for a workout').
    Use for vague or descriptive needs. Returns titles with price and rating."""
    docs = store.as_retriever(search_kwargs={"k": 4}).invoke(need)
    return "\n".join(
        f"{d.metadata['title'][:60]} (${d.metadata['price']}, {d.metadata['rating']}*)"
        for d in docs)


@tool
def keyword_search(terms: str) -> str:
    """Find products by exact words, brand, or model name in the listing (BM25).
    Use when the shopper names a specific product. Returns titles with price."""
    docs = store.as_bm25_retriever(k=4).invoke(terms)
    return "\n".join(
        f"{d.metadata['title'][:60]} (${d.metadata['price']}, {d.metadata['rating']}*)"
        for d in docs)


@tool
def query_catalog(sql: str) -> str:
    """Run a read-only SQL SELECT over the catalog for counts, averages, or filters.
    Table 'catalog' has columns: doc_id, page_content, price (float), rating (float).
    Example: SELECT COUNT(*) AS n FROM catalog WHERE price < 40 AND rating >= 4.5"""
    return str(db.query_sql(sql).to_pydict())

## The agent — tools + memory

`create_agent` wires the tools to the LLM; a `MemorySaver` checkpointer keyed by
`thread_id` gives the conversation memory, so follow-ups resolve against earlier
turns.

In [4]:
agent = create_agent(
    llm, [semantic_search, keyword_search, query_catalog],
    checkpointer=MemorySaver(),
)
CONFIG = {"configurable": {"thread_id": "shopper-1"}}


def ask(question: str) -> None:
    out = agent.invoke({"messages": [("user", question)]}, CONFIG)
    print(f"\nQ: {question}\nA: {out['messages'][-1].content[:280]}")


ask("I want wireless earbuds for running, budget around $50")     # -> semantic
ask("do you carry the NovaSound earbuds, and what's the price?")  # -> keyword
ask("how many products are rated 4.5 or higher?")                 # -> SQL
ask("of the highly rated ones, suggest a gift under $40")         # -> memory


Q: I want wireless earbuds for running, budget around $50
A: Here’s the best match I found for **wireless earbuds for running around $50**:

- **NovaSound Z9** — **$49.00**, **4.7★**

It appears to fit your budget and use case best from the results.

If you want, I can also help narrow further by priorities like:
- **most secure fit for ru



Q: do you carry the NovaSound earbuds, and what's the price?
A: Yes — **NovaSound Z9** is available in the catalog at **$49.00**.



Q: how many products are rated 4.5 or higher?
A: There are **139 products** rated **4.5 or higher**.



Q: of the highly rated ones, suggest a gift under $40
A: A good gift option under **$40** from the highly rated items is:

- **Victoria's Secret Love Spell and Pure Seduction Fragrance Lotion** — **$28.80**, **4.7★**

Another solid option:
- **LOWOSAIWOR Vintage 20s Headpiece for Women** — **$10.30**, **4.7★**

If you want, I can sugge


## Semantic cache — paraphrased repeats skip the LLM

`InfinoSemanticCache` stores prompts and their answers in the same engine and
returns a cached answer when a new prompt is *semantically* close (not just
identical). A paraphrase of an earlier question is served without another LLM
call. `score_threshold` is the cosine-distance cutoff for a hit — loosen it to
catch looser paraphrases, tighten it to require near-duplicates.

In [5]:
set_llm_cache(InfinoSemanticCache(
    db, embeddings, dim=DIM, table_name="llm_cache", score_threshold=0.25))

prompt = "In one sentence, what is BM25 search?"
paraphrase = "Briefly, in a single sentence, explain BM25 search."
t = time.time(); llm.invoke(prompt); first = time.time() - t
t = time.time(); llm.invoke(paraphrase); cached = time.time() - t
print(f"first call {first:.2f}s  ->  paraphrase {cached:.2f}s (served from cache)")

first call 1.58s  ->  paraphrase 0.03s (served from cache)


## Takeaway

One Infino store gave the agent semantic, keyword, and SQL search as tools, plus
a semantic LLM cache and conversation memory — all from one engine over one copy
of the data.